**MGMT298D: Science and Strategy of AI**

# Week 6B: Building a Tiny LLM

We build a small GPT-style language model and train it on Yelp reviews. The goal is to watch the model go from producing gibberish to coherent sentences as training progresses.

# 1 Setup

In [ ]:
#@title Import libraries
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML

---
# 2 Load Yelp Reviews

#### We grab 25,000 Yelp reviews and tokenize them into integer sequences. The training target is simple: given a sequence of tokens, predict the next one.

In [ ]:
dataset = load_dataset('yelp_review_full', split='train')

NUM_REVIEWS = 25000
texts = dataset['text'][:NUM_REVIEWS]

# Preview a sample of reviews
for i in range(50):
    stars = dataset['label'][i] + 1
    print(f"  [{stars}★] {texts[i][:100].replace(chr(10), ' ')}...")

In [ ]:
#@title Tokenize reviews
VOCAB_SIZE = 10000
SEQ_LEN = 64   # context window

# Convert raw text → integer token IDs
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=SEQ_LEN + 1)
vectorizer.adapt(texts)
vocab = vectorizer.get_vocabulary()

print(f"Vocabulary: {len(vocab):,} tokens")
print(f"Sample:     {vocab[:15]}")
print(f"\n'the food was great' → {vectorizer(['the food was great']).numpy()[0][:5]}")

In [ ]:
#@title Build training sequences (input → next token)
all_tokens = vectorizer(np.array(texts)).numpy()
all_tokens = all_tokens[np.sum(all_tokens > 0, axis=1) > 20]  # drop very short reviews

# Input = tokens[:-1], Target = tokens shifted by one position
# This is how every LLM is trained: predict the next token
x_train = all_tokens[:, :-1]
y_train = all_tokens[:, 1:]

print(f"Training sequences: {x_train.shape[0]:,}")

---
# 3 Build the Model

#### Same architecture as GPT, just much smaller. The key ingredient is **causal masking**: each token can only attend to previous tokens, so the model can't cheat by looking ahead.

In [ ]:
#@title Token + position embeddings
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, embed_dim)  # what word?
        self.pos_emb   = layers.Embedding(maxlen, embed_dim)      # where in the sentence?

    def call(self, x):
        return self.token_emb(x) + self.pos_emb(tf.range(tf.shape(x)[-1]))

In [ ]:
# Transformer block with causal ("can't look ahead") attention
class CausalTransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.ffn  = keras.Sequential([layers.Dense(ff_dim, activation='relu'), layers.Dense(embed_dim)])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x, training=False):
        attn = self.att(x, x, use_causal_mask=True)   # self-attention, no peeking forward
        x = self.norm1(x + attn)                       # residual + normalize
        return self.norm2(x + self.ffn(x))             # feed-forward + residual + normalize

In [ ]:
# Assemble the model
inputs  = layers.Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, 128)(inputs)
x = CausalTransformerBlock(128, num_heads=4, ff_dim=256)(x)    # block 1
x = CausalTransformerBlock(128, num_heads=4, ff_dim=256)(x)    # block 2
outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)    # predict next token

model = keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print(f"Parameters: {model.count_params():,}  (GPT-2 = 124M, GPT-4 ≈ 1.8T)")

---
# 4 Generation Helpers

#### Two utilities we'll reuse after each training phase. `show_predictions` displays the model's top guesses for the next token. The interactive widget lets you type any prompt and watch the model generate word by word — exactly how ChatGPT works.

In [ ]:
#@title Define generation utilities + interactive widget
id_to_word = dict(enumerate(vocab))

PROMPTS = ["the food was", "i would definitely",
           "the service at this restaurant", "we waited for"]


def show_predictions(model):
    """Print the model's top-5 next-token predictions for each prompt."""
    for prompt in PROMPTS:
        tokens = vectorizer([prompt]).numpy()[0]
        prompt_len = int(np.max(np.where(tokens > 0))) + 1
        padded = np.zeros(SEQ_LEN, dtype='int32')
        padded[:prompt_len] = tokens[:prompt_len]
        probs = model.predict(padded[np.newaxis, :], verbose=0)[0][prompt_len]
        top5 = np.argsort(probs)[-5:][::-1]
        preds = ', '.join(f"{id_to_word[t]} ({probs[t]:.3f})" for t in top5)
        print(f'  "{prompt}" → {preds}')


def generate_live(model, prompt, output_area, length=50, temperature=0.8):
    """Generate text word by word, updating the display after each token."""
    tokens = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(tokens > 0)[0]
    if len(nonzero) == 0:
        with output_area:
            print("(empty prompt — try typing something)")
        return
    out = list(tokens[:nonzero[-1] + 1])
    prompt_words = [id_to_word[t] for t in out]

    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype='int32')
        padded[:len(out)] = out[-SEQ_LEN:]
        p = model.predict(padded[np.newaxis, :], verbose=0)[0][min(len(out), SEQ_LEN-1)]
        p = np.exp(np.log(p + 1e-10) / temperature); p /= p.sum()
        t = np.random.choice(len(p), p=p)
        if t == 0: break
        out.append(t)

        # Update display: prompt in bold, generated words appearing one by one
        prompt_html = ' '.join(prompt_words)
        generated_words = [id_to_word[tok] for tok in out[len(prompt_words):]]
        gen_html = ' '.join(generated_words)
        output_area.clear_output(wait=True)
        with output_area:
            display(HTML(f'<span style="font-size:15px"><b>{prompt_html}</b> {gen_html}<span style="color:#999">▌</span></span>'))
        time.sleep(0.05)  # small delay so you can watch each word appear

    # Final display without cursor
    prompt_html = ' '.join(prompt_words)
    generated_words = [id_to_word[tok] for tok in out[len(prompt_words):]]
    gen_html = ' '.join(generated_words)
    output_area.clear_output(wait=True)
    with output_area:
        display(HTML(f'<span style="font-size:15px"><b>{prompt_html}</b> {gen_html}</span>'))


def make_generator_widget(model, title="Try it yourself"):
    """Create an interactive text box + generate button."""
    prompt_box = widgets.Text(
        value='the food was',
        placeholder='Type a prompt...',
        description='Prompt:',
        layout=widgets.Layout(width='500px'),
        style={'description_width': '60px'}
    )
    button = widgets.Button(description='Generate', button_style='primary')
    output_area = widgets.Output(layout=widgets.Layout(min_height='40px', padding='10px'))

    def on_click(b):
        output_area.clear_output()
        generate_live(model, prompt_box.value, output_area)

    def on_enter(change):
        output_area.clear_output()
        generate_live(model, prompt_box.value, output_area)

    button.on_click(on_click)
    prompt_box.on_submit(on_enter)

    display(HTML(f'<h4>{title}</h4>'))
    display(widgets.HBox([prompt_box, button]))
    display(output_area)

---
# 5 Phase 1 — Barely Trained (2 Epochs)

#### The model has barely seen any data. Its predictions should be nearly random.

In [ ]:
h1 = model.fit(x_train, y_train, batch_size=128, epochs=2, validation_split=0.05)

all_loss = list(h1.history['loss'])
all_val  = list(h1.history['val_loss'])
all_acc  = list(h1.history['accuracy'])

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 1 — Generate with the undertrained model")

Mostly incoherent — the model is guessing almost randomly.

---
# 6 Phase 2 — Getting Better (10 Epochs Total)

#### After more training the model should start picking up Yelp vocabulary and common phrases.

In [ ]:
h2 = model.fit(x_train, y_train, batch_size=128, epochs=8, validation_split=0.05)

all_loss += h2.history['loss']
all_val  += h2.history['val_loss']
all_acc  += h2.history['accuracy']

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 2 — Generate with the partially trained model")

Better — the model uses food and restaurant vocabulary in the right places.

---
# 7 Phase 3 — Coherent (25 Epochs Total)

#### With enough training, even this tiny model should produce recognizable Yelp-style sentences.

In [ ]:
h3 = model.fit(x_train, y_train, batch_size=128, epochs=15, validation_split=0.05)

all_loss += h3.history['loss']
all_val  += h3.history['val_loss']
all_acc  += h3.history['accuracy']

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 3 — Generate with the trained model")

Recognizable review-style text — food adjectives, service descriptions, recommendations. Still far from GPT-4, but the same fundamental mechanism.

---
# 8 Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(all_loss) + 1)

ax1.plot(epochs, all_loss, 'b-o', ms=3, label='Train')
ax1.plot(epochs, all_val, 'r-o', ms=3, label='Val')
ax1.axvline(2, color='gray', ls='--', alpha=.5)
ax1.axvline(10, color='gray', ls='--', alpha=.5)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss (lower = better predictions)')
ax1.legend()

ax2.plot(epochs, all_acc, 'b-o', ms=3)
ax2.axvline(2, color='gray', ls='--', alpha=.5)
ax2.axvline(10, color='gray', ls='--', alpha=.5)
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Next-Token Prediction Accuracy')

plt.tight_layout()
plt.show()

---
# 9 Takeaways

In [ ]:
print(f"{'Phase':<8s} {'Epochs':<9s} {'Loss':<10s} {'Accuracy':<10s} {'Output quality'}")
print("-" * 60)
print(f"{'1':<8s} {'2':<9s} {all_loss[1]:<10.4f} {all_acc[1]:<10.4f} {'Random word salad'}")
print(f"{'2':<8s} {'10':<9s} {all_loss[9]:<10.4f} {all_acc[9]:<10.4f} {'Right vocab, partial grammar'}")
print(f"{'3':<8s} {'25':<9s} {all_loss[-1]:<10.4f} {all_acc[-1]:<10.4f} {'Coherent review sentences'}")

print(f"\nThe trick behind every LLM: predict the next token.")
print(f"Our model: {model.count_params():,} params.  GPT-4: ~1.8 trillion.  Same idea.")